## Timestamp gambar yang di ambil masih belum sesuai dengan timestamp sensor 

In [6]:
from pathlib import Path
from datetime import datetime
import re
import shutil

import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment

# Lokasi input dan output
csv_file = Path(
    r"D:\00 Kuliah\TA\Experiment\Dataset\Data Sensor Kultivasi\csv\gabungan.csv"
)
image_dir = Path(
    r"D:\00 Kuliah\TA\Experiment\Dataset\Data Gambar\final_image"
)

output_image_dir = Path(
    r"D:\00 Kuliah\TA\Experiment\Dataset\Data Gambar\gambar_terpilih"
)
output_csv = csv_file.with_name("gabungan_dengan_gambar.csv")

maks_selisih_hari = 365

if not csv_file.is_file():
    raise FileNotFoundError(f"File CSV tidak ditemukan: {csv_file}")

if not image_dir.is_dir():
    raise NotADirectoryError(f"Folder gambar tidak ditemukan: {image_dir}")

output_image_dir.mkdir(parents=True, exist_ok=True)

# Baca data sensor
df = pd.read_csv(csv_file, encoding="utf-8-sig", dtype=str)

kolom_tanggal = "Tanggal"
kolom_waktu = "Waktu (jam:menit:detik)"

kolom_diperlukan = [kolom_tanggal, kolom_waktu]
kolom_tidak_ada = [kolom for kolom in kolom_diperlukan if kolom not in df.columns]

if kolom_tidak_ada:
    raise KeyError(f"Kolom berikut tidak ditemukan di CSV: {kolom_tidak_ada}")

# Gabungkan tanggal dan waktu sensor menjadi timestamp
tanggal = pd.to_datetime(df[kolom_tanggal], errors="coerce")

# Mendukung nilai waktu seperti "16:10:00" maupun "1900-01-01 16:10:00"
waktu_teks = df[kolom_waktu].astype(str).str.extract(
    r"(\d{1,2}:\d{2}(?::\d{2})?)",
    expand=False,
)

waktu_sensor = pd.to_datetime(
    tanggal.dt.strftime("%Y-%m-%d") + " " + waktu_teks,
    errors="coerce",
)

baris_invalid = waktu_sensor.isna()

if baris_invalid.any():
    nomor_baris = (df.index[baris_invalid] + 2).tolist()
    raise ValueError(
        f"Tanggal atau waktu tidak terbaca pada baris CSV: {nomor_baris}"
    )

# Baca timestamp dari nama file gambar, contohnya:
# ITS-POME_2025-10-20_17-00-05.jpg
ekstensi_gambar = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
pola_nama = re.compile(r"(\d{4}-\d{2}-\d{2})_(\d{2}-\d{2}-\d{2})")

gambar = []

for path in sorted(image_dir.rglob("*")):
    if not path.is_file() or path.suffix.lower() not in ekstensi_gambar:
        continue

    cocok = pola_nama.search(path.stem)
    if not cocok:
        continue

    timestamp_gambar = datetime.strptime(
        f"{cocok.group(1)} {cocok.group(2)}",
        "%Y-%m-%d %H-%M-%S",
    )
    gambar.append((pd.Timestamp(timestamp_gambar), path))

if not gambar:
    raise FileNotFoundError(
        f"Tidak ditemukan gambar dengan timestamp yang dikenali di: {image_dir}"
    )

if len(gambar) < len(df):
    raise ValueError(
        f"Jumlah gambar ({len(gambar)}) lebih sedikit daripada "
        f"jumlah baris sensor ({len(df)}), jadi semua sensor tidak bisa "
        "mendapat gambar unik."
    )

# Hitung selisih waktu setiap sensor terhadap setiap gambar, dalam detik
sensor_ns = waktu_sensor.astype("int64").to_numpy()
gambar_ns = np.array(
    [timestamp.value for timestamp, _ in gambar],
    dtype=np.int64,
)

selisih_detik = np.abs(
    sensor_ns[:, None] - gambar_ns[None, :]
) / 1_000_000_000

# Pasangkan secara global: satu gambar hanya bisa dipakai satu kali.
# Jika jumlah gambar mencukupi, setiap sensor akan mendapat satu gambar.
baris_sensor, kolom_gambar = linear_sum_assignment(selisih_detik)

if len(baris_sensor) != len(df):
    raise RuntimeError(
        f"Hanya {len(baris_sensor)} dari {len(df)} baris sensor yang mendapat pasangan."
    )

nama_gambar = [""] * len(df)
selisih_waktu = [""] * len(df)
gambar_terpilih = []

batas_selisih = pd.Timedelta(days=maks_selisih_hari)

for indeks_sensor, indeks_gambar in zip(baris_sensor, kolom_gambar):
    timestamp_gambar, path_gambar = gambar[indeks_gambar]
    delta = abs(waktu_sensor.iloc[indeks_sensor] - timestamp_gambar)

    if delta > batas_selisih:
        raise ValueError(
            f"Pasangan untuk baris CSV {indeks_sensor + 2} melewati "
            f"batas {maks_selisih_hari} hari: {delta}"
        )

    nama_gambar[indeks_sensor] = path_gambar.name
    selisih_waktu[indeks_sensor] = str(delta)
    gambar_terpilih.append(path_gambar)

# Pastikan setiap sensor mendapat gambar berbeda
if len(set(gambar_terpilih)) != len(df):
    raise RuntimeError("Ada gambar yang terpasang ke lebih dari satu sensor.")

# Tambahkan nama gambar dan selisih waktunya ke CSV
df["Nama_Gambar"] = nama_gambar
df["Selisih_Waktu"] = selisih_waktu

# Salin gambar terpilih ke folder output
for path_gambar in gambar_terpilih:
    shutil.copy2(path_gambar, output_image_dir / path_gambar.name)

# Simpan CSV hasil
df.to_csv(output_csv, index=False, encoding="utf-8-sig")

print(f"Baris sensor: {len(df)}")
print(f"Sensor yang mendapat gambar: {len(gambar_terpilih)}")
print(f"Jumlah gambar unik yang dipakai: {len(set(gambar_terpilih))}")
print(f"CSV hasil: {output_csv}")
print(f"Gambar terpilih disalin ke: {output_image_dir}")

Baris sensor: 101
Sensor yang mendapat gambar: 101
Jumlah gambar unik yang dipakai: 101
CSV hasil: D:\00 Kuliah\TA\Experiment\Dataset\Data Sensor Kultivasi\csv\gabungan_dengan_gambar.csv
Gambar terpilih disalin ke: D:\00 Kuliah\TA\Experiment\Dataset\Data Gambar\gambar_terpilih
